## Processamento Inicial

Nesse notebook faremos um processamento inicial do dataset bruto, convertendo os tipos das features adequadamente para inteiros, floats, tipos categóricos e datas. Além disso faremos uma etapa de sanitização para converter dados inconsistentes em ´NA´ e salvando o dataframe resultante em formato parquet.

### Importando as Bibliotecas

Primeiro vamos importar as bibliotecas que usaremos.

In [24]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### Carregando o Dataset

Agora vamos carregar o dataset.

In [25]:
df = pd.read_csv("../data/raw/UCMF_raw.csv")

### Análise Inicial

Vamos fazer um análise inicial, olhando para um trecho do `DataFrame` e analisando as informações mais básicas dele.

In [26]:
df.head()

,ID,Peso,Altura,IMC,Atendimento,DN,IDADE,Convenio,PULSOS,PA SISTOLICA,...,PPA,NORMAL X ANORMAL,B2,SOPRO,FC,HDA 1,HDA2,SEXO,MOTIVO1,MOTIVO2
0,1,5.0,51,19.0,11/05/06,30/03/06,0.12,GS,Normais,NaN,...,Não Calculado,Anormal,Normal,Sistólico,112,Palpitacao,NaN,M,6 - Suspeita de cardiopatia,6 - Palpitação/taquicardia/arritmia
1,2,3.5,50,14.0,25/05/05,19/05/05,0.02,GS,Normais,NaN,...,Não Calculado,Anormal,Normal,ausente,128,Dispneia,NaN,M,6 - Suspeita de cardiopatia,6 - Dispnéia
2,3,0.0,0,NaN,12/06/01,08/05/05,-4.05,SULA,Normais,NaN,...,Não Calculado,Anormal,Normal,Sistólico,88,Assintomático,NaN,M,2 - Check-up,NaN
3,4,8.1,65,19.0,15/10/09,21/04/09,0.50,NaN,Normais,NaN,...,Não Calculado,Anormal,Normal,ausente,92,Assintomático,NaN,M,5 - Parecer cardiológico,NaN
4,7,40.0,151,18.0,14/01/08,14/08/95,12.89,SAME,Normais,NaN,...,Não Calculado,Anormal,Normal,ausente,96,Dor precordial,NaN,M,5 - Parecer cardiológico,NaN


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                12873 non-null  int64  
 1   Peso              12555 non-null  float64
 2   Altura            12873 non-null  int64  
 3   IMC               8146 non-null   float64
 4   Atendimento       11890 non-null  str    
 5   DN                11497 non-null  str    
 6   IDADE             11377 non-null  float64
 7   Convenio          8765 non-null   str    
 8   PULSOS            11679 non-null  str    
 9   PA SISTOLICA      5143 non-null   float64
 10  PA DIASTOLICA     5133 non-null   float64
 11  PPA               12656 non-null  str    
 12  NORMAL X ANORMAL  11705 non-null  str    
 13  B2                11695 non-null  str    
 14  SOPRO             11706 non-null  str    
 15  FC                11005 non-null  str    
 16  HDA 1             8603 non-null   str    
 17  HDA2

Nosso dataset tem, inicialmente, $21$ features e $12873$ registros, sendo a feature `"NORMAL X ANORMAL"` o nosso target.

Como podemos ver alguns nomes de features estão em maiúsculo e com espaços. Vamos então limpar esses nomes e adotar a seguinte convenção: nomes com letras minúsculas e underscore (`_`) como separador. Além disso vamos renomear as features `"DN"`, `"HDA 1"` e `"NORMAL X ANORMAL"` em `"nascimento"`, `"hda1"` e `"patologia"`, respectivamente.

In [28]:
def normalize_names(x):
    return "_".join(x.strip().lower().split())

df = (df
    .rename(lambda x: normalize_names(x), axis="columns")
    .rename({
        "dn": "nascimento",
        "hda_1": "hda1",
        "normal_x_anormal": "patologia",
    }, axis="columns")
)

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             12873 non-null  int64  
 1   peso           12555 non-null  float64
 2   altura         12873 non-null  int64  
 3   imc            8146 non-null   float64
 4   atendimento    11890 non-null  str    
 5   nascimento     11497 non-null  str    
 6   idade          11377 non-null  float64
 7   convenio       8765 non-null   str    
 8   pulsos         11679 non-null  str    
 9   pa_sistolica   5143 non-null   float64
 10  pa_diastolica  5133 non-null   float64
 11  ppa            12656 non-null  str    
 12  patologia      11705 non-null  str    
 13  b2             11695 non-null  str    
 14  sopro          11706 non-null  str    
 15  fc             11005 non-null  str    
 16  hda1           8603 non-null   str    
 17  hda2           407 non-null    str    
 18  sexo           12

Agora temos nomes mais apropriados para as features e seguindo uma convenção adequada. Vamos então analisar feature por feature para convertê-las em tipos mais apropriadas e garantir a consistência dos dados.

### Features Temporais (Datas)

Agora, vamos processar as features temporais `"atendimento"` e `"nascimento"`. Ambas as features são do tipo string (`str`), portanto precisamos convertê-las para o tipo data (`datetime`). Vamos dar uma olhadinha nos formatos de datas que temos.

In [30]:
cols = ["atendimento", "nascimento"]

df[cols].head()

,atendimento,nascimento
0,11/05/06,30/03/06
1,25/05/05,19/05/05
2,12/06/01,08/05/05
3,15/10/09,21/04/09
4,14/01/08,14/08/95


Analisando esse trecho do `DataFrame`, podemos inferir que os dados estão no formato `"%d/%m/%y"`. Para verificar isso vamos ver quantos registros respeitam o padrão `\d{2}/\d{2}/\d{2}` (dois digitos para o dia, mês e ano) de regex no Python.

In [31]:
for col in cols:
    mask = df[col].str.match(r"^\d{2}/\d{2}/\d{2}$")
    mached = df[mask][col]
    count = mached.shape[0]

    print(f"{col}: {count}")

atendimento: 11578
nascimento: 11259


Como podemos ver, a maioria das datas estão realmente no padrão de regex proposto e por isso vamos converter essas features para `datetime`. Note que usamos o argumento `errors="coerce"` para que o pandas salve as datas que não puderam ser convertidas como `pd.NaT`.

In [32]:
format = "%d/%m/%y"

for col in cols:
    df[col] = pd.to_datetime(df[col], format=format, errors="coerce")

Agora, vamos checar se nossa conversão foi bem sucedida.

In [33]:
df[cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   atendimento  11578 non-null  datetime64[us]
 1   nascimento   11253 non-null  datetime64[us]
dtypes: datetime64[us](2)
memory usage: 201.3 KB


Como podemos notar, temos $11578$ e $11253$ datas convertidas nas features `"atendimento"` e `"nascimento"`, respectivamente. Então, vamos checar algumas estatísticas dessas datas.

In [34]:
df[cols].describe()

,atendimento,nascimento
count,11578,11253
mean,2007-03-11 06:01:40.742788,2002-02-26 14:47:18.922953
min,1990-01-03 00:00:00,1969-07-14 00:00:00
25%,2005-07-07 00:00:00,1998-08-15 00:00:00
50%,2007-05-30 00:00:00,2003-01-24 00:00:00
75%,2009-01-05 00:00:00,2006-03-27 00:00:00
max,2066-08-22 00:00:00,2068-07-06 00:00:00


Note como a maioria das datas estão entre os anos $1990$ e $2009$, mas algumas datas são de anos acima de $2026$, o que na prática é impossível já que essas datas são de events que já ocorreram. Vamos investigar isso.

In [35]:
nascimentos = df["nascimento"].copy()
atendimentos = df["atendimento"].copy()

nascimentos = (nascimentos.dt.year
    .sort_values()
    .value_counts(normalize=True, sort=False)
)

atendimentos = (atendimentos.dt.year
    .sort_values()
    .value_counts(normalize=True, sort=False)
)

print(nascimentos)
print(atendimentos)

nascimento
1969.0    0.000178
1970.0    0.000089
1971.0    0.000089
1972.0    0.000089
1973.0    0.000089
1974.0    0.000089
1975.0    0.000178
1976.0    0.000089
1977.0    0.000089
1978.0    0.000355
1979.0    0.000267
1980.0    0.000355
1981.0    0.000622
1982.0    0.000355
1983.0    0.000800
1984.0    0.000533
1985.0    0.000800
1986.0    0.001066
1987.0    0.002222
1988.0    0.003466
1989.0    0.005687
1990.0    0.008620
1991.0    0.011019
1992.0    0.017151
1993.0    0.019728
1994.0    0.031014
1995.0    0.033680
1996.0    0.036968
1997.0    0.042566
1998.0    0.048432
1999.0    0.049853
2000.0    0.054030
2001.0    0.061584
2002.0    0.062917
2003.0    0.082645
2004.0    0.077046
2005.0    0.075713
2006.0    0.079445
2007.0    0.069759
2008.0    0.067626
2009.0    0.047809
2010.0    0.003821
2022.0    0.000089
2027.0    0.000089
2036.0    0.000089
2040.0    0.000089
2051.0    0.000089
2055.0    0.000178
2064.0    0.000089
2065.0    0.000089
2068.0    0.000267
Name: proportion, dt

Como podemos observar, as datas estão, em sua maioria entre $1990$ e $2010$. Portanto vamos manter as datas nesse intervalo e as datas que estiverem fora nós vamos converter para `pd.NaT`.

In [36]:
for col in cols:
    df[col] = df[col].where(df[col].dt.year <= 2023)

In [37]:
df[cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   atendimento  11577 non-null  datetime64[us]
 1   nascimento   11242 non-null  datetime64[us]
dtypes: datetime64[us](2)
memory usage: 201.3 KB


Agora podemos olhar as infos do nosso dataset e ver que as features temporais já estão com seus tipos convertidos.

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             12873 non-null  int64         
 1   peso           12555 non-null  float64       
 2   altura         12873 non-null  int64         
 3   imc            8146 non-null   float64       
 4   atendimento    11577 non-null  datetime64[us]
 5   nascimento     11242 non-null  datetime64[us]
 6   idade          11377 non-null  float64       
 7   convenio       8765 non-null   str           
 8   pulsos         11679 non-null  str           
 9   pa_sistolica   5143 non-null   float64       
 10  pa_diastolica  5133 non-null   float64       
 11  ppa            12656 non-null  str           
 12  patologia      11705 non-null  str           
 13  b2             11695 non-null  str           
 14  sopro          11706 non-null  str           
 15  fc             11005 non-null 

## Feature Numéricas

Vamos tratar as features numéricas. Para isso vamos tratar as features `"peso"`, `"altura"`, `"idade"` e `"imc"` e mais tarde trataremos das features `"pa_sistolica"` e `"pa_diastolica"`.

Primeiro vamos olhar para algumas estatísticas dos dados de `"peso"`, `"altura"` e `"idade"`.

In [39]:
cols = ["peso", "altura", "idade"]

df[cols].describe()

,peso,altura,idade
count,12555.000000,12873.000000,11377.000000
mean,16.614417,66.111474,4.856947
std,16.979814,56.027934,8.829742
min,-40.000000,0.000000,-113.180000
25%,3.400000,0.000000,0.930000
50%,12.800000,69.000000,3.960000
75%,24.000000,112.000000,8.590000
max,157.000000,198.000000,71.810000


Como podemos ver, temos alguns registros negativos em `"peso"` e `"idade"`. Portanto vamos converter os dados negativos para `np.nan` e garantir que as $3$ features tenham tipo `np.float64`.

In [40]:
for col in cols:
    df[col] = df[col].astype(np.float64)
    df[col] = df[col].where(df[col] >= 0.0)

df[cols].describe()

,peso,altura,idade
count,12554.000000,12873.000000,11255.000000
mean,16.618926,66.111474,5.374211
std,16.972970,56.027934,5.129182
min,0.000000,0.000000,0.000000
25%,3.400000,0.000000,1.025000
50%,12.800000,69.000000,4.040000
75%,24.000000,112.000000,8.630000
max,157.000000,198.000000,71.810000


In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             12873 non-null  int64         
 1   peso           12554 non-null  float64       
 2   altura         12873 non-null  float64       
 3   imc            8146 non-null   float64       
 4   atendimento    11577 non-null  datetime64[us]
 5   nascimento     11242 non-null  datetime64[us]
 6   idade          11255 non-null  float64       
 7   convenio       8765 non-null   str           
 8   pulsos         11679 non-null  str           
 9   pa_sistolica   5143 non-null   float64       
 10  pa_diastolica  5133 non-null   float64       
 11  ppa            12656 non-null  str           
 12  patologia      11705 non-null  str           
 13  b2             11695 non-null  str           
 14  sopro          11706 non-null  str           
 15  fc             11005 non-null 

Por fim, podemos notar que `"peso"` está em quilos ($\text{kg}$), `"altura"` está em centimetros ($\text{cm}$) e `"idade"` está em anos.